# Udaplay Project Solution
## Part 01 - Offline RAG

This notebook builds a persistent Chroma collection from the provided game JSON files and demonstrates semantic retrieval over the indexed dataset.

Expected environment variables in `.env`:
- `OPENAI_API_KEY`
- `CHROMA_OPENAI_API_KEY`
- `TAVILY_API_KEY`

If an OpenAI embedding key is missing, the helper service falls back to Chroma's default embedding function so the notebook still remains usable.

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / 'starter').exists():
    project_root = project_root.parent

starter_path = project_root / 'starter'
if str(starter_path) not in sys.path:
    sys.path.append(str(starter_path))

from lib.udaplay import UdaPlayServices

In [ ]:
load_dotenv(project_root / '.env')

services = UdaPlayServices(
    games_directory=str(project_root / 'starter' / 'games'),
    persist_directory=str(project_root / 'starter' / 'chromadb'),
    collection_name='udaplay_default',
    memory_collection_name='udaplay_research_memory',
    openai_api_key=None,
    tavily_api_key=os.getenv('TAVILY_API_KEY'),
)
embedding_source = 'Chroma default embeddings'
print(f'Project root: {project_root}')
print(f'Games directory: {services.games_directory}')
print(f'Persistent store: {services.persist_directory}')
print(f'Embedding source: {embedding_source}')

Project root: c:\Users\modassir.hussain\Downloads\project3
Games directory: c:\Users\modassir.hussain\Downloads\project3\starter\games
Persistent store: c:\Users\modassir.hussain\Downloads\project3\starter\chromadb
Embedding source: Chroma default embeddings


In [ ]:
records = services.load_games()
print(f'Loaded {len(records)} game JSON files')
print(json.dumps(records[0].to_document().metadata, indent=2))

Loaded 15 game JSON files
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997,
  "source_path": "c:\\Users\\modassir.hussain\\Downloads\\project3\\starter\\games\\001.json"
}


In [ ]:
indexed_count = services.index_games(force=True)
collection_snapshot = services.vector_store.get()
print(f'Indexed {indexed_count} games into collection: {services.collection_name}')
print(f'Collection size: {len(collection_snapshot.get("ids", []))}')

Indexed 15 games into collection: udaplay_default
Collection size: 15


In [ ]:
preview = services.vector_store.get(limit=3)
for doc_id, metadata in zip(preview.get('ids', []), preview.get('metadatas', [])):
    print(f'ID: {doc_id}')
    print(json.dumps(metadata, indent=2))
    print('-' * 60)

ID: 001
{
  "Name": "Gran Turismo",
  "source_path": "c:\\Users\\modassir.hussain\\Downloads\\project3\\starter\\games\\001.json",
  "YearOfRelease": 1997,
  "Publisher": "Sony Computer Entertainment",
  "Genre": "Racing",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "Platform": "PlayStation 1"
}
------------------------------------------------------------
ID: 002
{
  "YearOfRelease": 2004,
  "Platform": "PlayStation 2",
  "source_path": "c:\\Users\\modassir.hussain\\Downloads\\project3\\starter\\games\\002.json",
  "Name": "Grand Theft Auto: San Andreas",
  "Description": "An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.",
  "Genre": "Action-adventure",
  "Publisher": "Rockstar Games"
}
------------------------------------------------------------
ID: 003
{
  "Platform": "PlayStation 3",
  "YearOfRelease": 2010,
  "source_path": "c:\\Us

In [ ]:
def print_top_hits(query: str, n_results: int = 3) -> None:
    """Run semantic retrieval and print a compact result summary."""
    result = services.retrieve_game(query=query, n_results=n_results)
    print(f'\nQuery: {query}')
    for item in result['results']:
        metadata = item['metadata']
        print(
            f"- score={item['score']:.3f} | {metadata['Name']} | {metadata['Platform']} | {metadata['YearOfRelease']} | {metadata['Publisher']}"
        )

sample_queries = [
    'When was Pokemon Gold and Silver released?',
    'Which Mario game was the first 3D platformer?',
    'Was Mortal Kombat X released for PlayStation 5?',
]

for question in sample_queries:
    print_top_hits(question)


Query: When was Pokemon Gold and Silver released?
- score=0.579 | Pokémon Gold and Silver | Game Boy Color | 1999 | Nintendo
- score=0.458 | Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | Nintendo
- score=0.404 | Super Mario 64 | Nintendo 64 | 1996 | Nintendo

Query: Which Mario game was the first 3D platformer?
- score=0.623 | Super Mario 64 | Nintendo 64 | 1996 | Nintendo
- score=0.554 | Super Mario World | Super Nintendo Entertainment System (SNES) | 1990 | Nintendo
- score=0.505 | Mario Kart 8 Deluxe | Nintendo Switch | 2017 | Nintendo

Query: Was Mortal Kombat X released for PlayStation 5?
- score=0.479 | Marvel's Spider-Man 2 | PlayStation 5 | 2023 | Sony Interactive Entertainment
- score=0.471 | Halo Infinite | Xbox Series X|S | 2021 | Xbox Game Studios
- score=0.465 | Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | Rockstar Games


The persistent vector store is now ready for the agent notebook. The key deliverables for Part 1 are covered here: local JSON ingestion, document formatting, persistent Chroma indexing, and semantic retrieval queries over the indexed collection.